# Wikidata Query Agent Evaluation

This notebook implements a Wikidata Query Agent using LangGraph for evaluation, using Qwen 7B (4-bit) as the LLM backend instead of Google Gemini.

The agent uses a graph-based approach to:
1. Extract entities and properties from natural language questions
2. Generate SPARQL queries for Wikidata
3. Execute and validate the queries
4. Evaluate performance against a test dataset

## Install Dependencies

In [1]:
!pip install -q langgraph unsloth langchain-huggingface tqdm sparqlwrapper langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.2/148.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.7/192.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.2/437.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Import Libraries

In [2]:
import os
import re
import json
import logging
import pandas as pd
from tqdm import tqdm
import requests
from typing import Dict, Any, List, Tuple, TypedDict, Annotated, Literal, Union
import langchain_huggingface.llms.huggingface_pipeline as _hf_mod
from SPARQLWrapper import SPARQLWrapper, JSON

_hf_mod.Union = Union

# Unsloth and HuggingFace imports
from langchain_huggingface import HuggingFacePipeline
from unsloth import FastLanguageModel
from transformers import pipeline

# LangGraph imports
import langgraph.graph as lg
from langgraph.graph import StateGraph, END

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-04-26 05:37:46.402926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745645866.578724      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745645866.632351      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Utility Tools

These tools help the agent interact with Wikidata.

In [3]:
class WikidataSearchTool:
    """Tool for searching entities or properties in Wikidata."""
    
    def __init__(self):
        logger.info("Initializing WikidataSearchTool")
    
    def search(self, term: str, type: Literal["entity", "property"] = "entity", limit: int = 5) -> List[Dict[str, Any]]:
        """
        Search for entities or properties in Wikidata
        
        Args:
            term: The search term
            type: Type of search ("entity" or "property")
            limit: Maximum number of results to return
            
        Returns:
            A list of matching entities or properties with their details
        """
        logger.info(f"WikidataSearchTool: Searching for {type} with term '{term}', limit={limit}")
        
        if type == "entity":
            return self._search_entity(term, limit)
        elif type == "property":
            return self._search_property(term, limit)
        else:
            logger.error(f"WikidataSearchTool: Invalid search type: {type}")
            raise ValueError(f"Invalid search type: {type}. Must be 'entity' or 'property'")
    
    def _search_entity(self, term: str, limit: int) -> List[Dict[str, Any]]:
        url = "https://www.wikidata.org/w/api.php"
        params = {
            "action": "wbsearchentities",
            "format": "json",
            "language": "en",
            "search": term,
            "limit": limit
        }
        
        logger.info(f"WikidataSearchTool: Calling Wikidata API to search for entity '{term}'")
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()
            
            results = []
            for item in data.get("search", []):
                result = {
                    "id": item.get("id"),
                    "label": item.get("label", ""),
                    "description": item.get("description", ""),
                    "url": item.get("url", "")
                }
                results.append(result)
            
            logger.info(f"WikidataSearchTool: Found {len(results)} entity results for '{term}'")
            if results:
                logger.debug(f"WikidataSearchTool: First result: {results[0]['label']} ({results[0]['id']})")
            
            return results
            
        except requests.exceptions.RequestException as e:
            logger.error(f"WikidataSearchTool: Error searching for entity '{term}': {str(e)}")
            return []
    
    def _search_property(self, term: str, limit: int) -> List[Dict[str, Any]]:
        url = "https://www.wikidata.org/w/api.php"
        params = {
            "action": "wbsearchentities",
            "format": "json",
            "language": "en",
            "search": term,
            "type": "property",
            "limit": limit
        }
        
        logger.info(f"WikidataSearchTool: Calling Wikidata API to search for property '{term}'")
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()
            
            results = []
            for item in data.get("search", []):
                result = {
                    "id": item.get("id"),
                    "label": item.get("label", ""),
                    "description": item.get("description", ""),
                    "url": item.get("url", "")
                }
                results.append(result)
            
            logger.info(f"WikidataSearchTool: Found {len(results)} property results for '{term}'")
            if results:
                logger.debug(f"WikidataSearchTool: First result: {results[0]['label']} ({results[0]['id']})")
                
            return results
            
        except requests.exceptions.RequestException as e:
            logger.error(f"WikidataSearchTool: Error searching for property '{term}': {str(e)}")
            return []

class WikidataSPARQLTool:
    """Tool for executing SPARQL queries against Wikidata."""
    
    def __init__(self):
        logger.info("Initializing WikidataSPARQLTool")
        self._endpoint = "https://query.wikidata.org/sparql"
        self._sparql = SPARQLWrapper(self._endpoint)
        self._sparql.setReturnFormat(JSON)
        # Set a user agent to be respectful to the Wikidata service
        self._sparql.addCustomHttpHeader("User-Agent", "LangGraph Wikidata Agent/1.0")
        
    def execute(self, query: str, limit: int = 5) -> Dict[str, Any]:
        """
        Execute a SPARQL query against Wikidata
        
        Args:
            query: The SPARQL query string
            limit: Maximum number of results to return
            
        Returns:
            The query results or error information
        """
        logger.info(f"WikidataSPARQLTool: Executing SPARQL query with limit={limit}")
        logger.debug(f"WikidataSPARQLTool: Query to execute:\n{query}")
        
        try:
            # Convert limit to integer explicitly to avoid float notation
            limit_value = int(limit)
            
            # Add limit if not already present in the query
            if "LIMIT" not in query.upper():
                logger.info("WikidataSPARQLTool: Adding LIMIT clause to query")
                query += f" LIMIT {limit_value}"
            
            self._sparql.setQuery(query)
            logger.info("WikidataSPARQLTool: Sending query to Wikidata SPARQL endpoint")
            results = self._sparql.query().convert()
            logger.info("WikidataSPARQLTool: Query executed successfully")
            
            # Process results to make them more readable
            processed_results = []
            
            if "results" in results and "bindings" in results["results"]:
                bindings = results["results"]["bindings"]
                logger.info(f"WikidataSPARQLTool: Processing {len(bindings)} results")
                
                for binding in bindings:
                    processed_binding = {}
                    for key, value in binding.items():
                        processed_binding[key] = value.get("value", "")
                    processed_results.append(processed_binding)
                
                return {
                    "success": True,
                    "results": processed_results,
                    "count": len(processed_results),
                    "raw_results": bindings  # Include raw results for reference
                }
            else:
                # Handle other types of results (e.g., ASK queries)
                logger.info("WikidataSPARQLTool: Query returned non-standard results structure")
                return {
                    "success": True,
                    "results": results,
                    "count": 1
                }
        
        except Exception as e:
            error_msg = str(e).split('\n')[0]
            logger.error(f"WikidataSPARQLTool: Query execution failed: {error_msg}")
            return {
                "success": False,
                "error": error_msg,
                "query": query
            }
            
    def get_labels_for_uris(self, uris: List[str]) -> Dict[str, str]:
        """
        Get human-readable labels for Wikidata URIs
        
        Args:
            uris: List of Wikidata entity URIs
            
        Returns:
            Dictionary mapping URIs to their labels
        """
        if not uris:
            return {}
            
        logger.info(f"WikidataSPARQLTool: Getting labels for {len(uris)} URIs")
        
        # Limit the number of URIs to process to avoid overly large queries
        if len(uris) > 50:
            logger.warning(f"WikidataSPARQLTool: Limiting label lookup to 50 URIs (out of {len(uris)})")
            uris = uris[:50]
        
        # Extract entity IDs from URIs
        entity_ids = []
        for uri in uris:
            if uri.startswith("http://www.wikidata.org/entity/"):
                entity_id = uri.split("/")[-1]
                entity_ids.append(entity_id)
                
        if not entity_ids:
            logger.warning("WikidataSPARQLTool: No valid entity IDs extracted from URIs")
            return {}
            
        # Construct VALUES clause for SPARQL query
        values_str = " ".join([f"wd:{entity_id}" for entity_id in entity_ids])
        
        # Construct SPARQL query to get labels
        query = f"""
        SELECT ?entity ?label WHERE {{
          VALUES ?entity {{ {values_str} }}
          ?entity rdfs:label ?label .
          FILTER(LANG(?label) = "en")
        }}
        """
        
        try:
            logger.info("WikidataSPARQLTool: Executing label lookup query")
            self._sparql.setQuery(query)
            results = self._sparql.query().convert()
            
            # Process results into a dictionary
            uri_to_label = {}
            if "results" in results and "bindings" in results["results"]:
                bindings = results["results"]["bindings"]
                logger.info(f"WikidataSPARQLTool: Found labels for {len(bindings)} entities")
                
                for binding in bindings:
                    if "entity" in binding and "value" in binding["entity"]:
                        entity_uri = binding["entity"]["value"]
                        label = binding["label"]["value"] if "label" in binding and "value" in binding["label"] else "Unknown"
                        uri_to_label[entity_uri] = label
            
            return uri_to_label
            
        except Exception as e:
            logger.error(f"WikidataSPARQLTool: Error getting labels: {str(e)}")
            return {}

## Initialize the Qwen LLM

In [4]:
def create_qwen_llm(max_seq_length=2048, load_in_4bit=True, dtype=None):
    """
    Initialize the Qwen 7B model with 4-bit quantization
    
    Args:
        max_seq_length: Maximum sequence length for the model
        load_in_4bit: Whether to load the model in 4-bit precision
        dtype: Data type for the model weights
        
    Returns:
        HuggingFacePipeline LLM object
    """
    print("Initializing Qwen 2.5 7B 4-bit model...")
            
    # Load tokenizer and model
    model_id = "Qwen/Qwen2.5-7B"  # Using the base model instead of Coder for better generalization
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    
    # Create text generation pipeline with parameters tuned for SPARQL generation
    text_generation_pipeline = pipeline(
        model=model,
        tokenizer=tokenizer,
        task="text-generation",
        max_new_tokens=1024,  # Increased for longer reasoning chains
        temperature=0.1,      # Lower temperature for more focused outputs
        repetition_penalty=1.1,
        return_full_text=False,
        trust_remote_code=True
    )
    
    # Create LangChain wrapper for the pipeline
    llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
    print("Qwen model initialized successfully.")
    return llm

## Graph Nodes

In [5]:
class EntityExtractor:
    """Node for extracting entities and properties from a user query."""

    def __init__(self, llm):
        logger.info("Initializing EntityExtractor")
        self.llm = llm
        self.search_tool = WikidataSearchTool()

    def extract_entities_properties(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Extract entities and properties from a user query

        Args:
            state: Current state with 'question' key

        Returns:
            Updated state with entities and properties
        """
        question = state.get("question", "")
        logger.info(f"EntityExtractor: Starting extraction for question: '{question}'")

        # Use Qwen to identify potential entities and properties
        prompt = f"""
        Analyze the following question and identify the key entities and properties needed to create a Wikidata SPARQL query.
        
        Question: {question}
        
        For each entity or property, provide:
        1. Term: The name of the entity or property
        2. Type: Whether it's an "entity" or "property"
        3. Importance: "required" or "optional"
        
        Format your response as a structured list of JSON objects:
        [
          {{"term": "entity name", "type": "entity", "importance": "required"}},
          {{"term": "property name", "type": "property", "importance": "required"}}
        ]
        
        Only respond with the JSON list, nothing else.
        """

        logger.info("EntityExtractor: Calling Qwen model to identify entities and properties")
        response_text = self.llm.invoke(prompt)
        logger.debug(f"EntityExtractor: Qwen response: {response_text[:500]}...")

        # Extract JSON from response
        try:
            # Try to parse the entire response as JSON first
            terms = json.loads(response_text)
            logger.info(f"EntityExtractor: Successfully parsed {len(terms)} terms from JSON response")
        except json.JSONDecodeError:
            # If that fails, try to extract JSON using regex
            logger.warning("EntityExtractor: Failed to parse JSON directly, attempting regex extraction")
            json_pattern = r"\[\s*\{.*\}\s*\]"
            json_match = re.search(json_pattern, response_text, re.DOTALL)

            if json_match:
                json_text = json_match.group(0)
                try:
                    terms = json.loads(json_text)
                    logger.info(f"EntityExtractor: Successfully extracted {len(terms)} terms using regex")
                except:
                    # Fallback if JSON parsing fails
                    logger.warning("EntityExtractor: JSON parsing failed even with regex, using fallback parser")
                    terms = self._parse_terms_fallback(response_text)
            else:
                logger.warning("EntityExtractor: No JSON-like pattern found, using fallback parser")
                terms = self._parse_terms_fallback(response_text)

        # Search Wikidata for each term
        entities = []
        properties = []

        logger.info(f"EntityExtractor: Searching Wikidata for {len(terms)} terms")
        for term in terms:
            term_name = term.get("term", "")
            term_type = term.get("type", "entity")
            importance = term.get("importance", "required")

            if not term_name:
                continue

            logger.info(f"EntityExtractor: Searching for {term_type} '{term_name}'")
            search_results = self.search_tool.search(term_name, term_type)

            if search_results:
                result = search_results[0]  # Take the top result
                result["original_term"] = term_name
                result["importance"] = importance

                if term_type == "entity":
                    logger.info(f"EntityExtractor: Found entity: {result['label']} ({result['id']})")
                    entities.append(result)
                else:
                    logger.info(f"EntityExtractor: Found property: {result['label']} ({result['id']})")
                    properties.append(result)
            else:
                logger.warning(f"EntityExtractor: No results found for {term_type} '{term_name}'")

        logger.info(f"EntityExtractor: Extraction complete. Found {len(entities)} entities and {len(properties)} properties")
        
        return {
            **state,
            "entities": entities,
            "properties": properties,
            "extraction_complete": True,
        }

    def _parse_terms_fallback(self, text: str) -> List[Dict[str, Any]]:
        """Fallback method to parse terms if JSON parsing fails."""
        logger.info("EntityExtractor: Using fallback parser for terms")
        terms = []
        lines = text.split("\n")

        current_term = {}
        for line in lines:
            line = line.strip()
            if not line:
                continue

            if "term:" in line.lower() or "entity:" in line.lower():
                if current_term and "term" in current_term:
                    terms.append(current_term)
                current_term = {}

                # Try to extract term
                match = re.search(r"[\"']([^\"']+)[\"']", line)
                if match:
                    current_term["term"] = match.group(1)
                else:
                    parts = line.split(":", 1)
                    if len(parts) > 1:
                        current_term["term"] = parts[1].strip()

            if "type:" in line.lower():
                if "entity" in line.lower():
                    current_term["type"] = "entity"
                elif "property" in line.lower():
                    current_term["type"] = "property"

            if "importance:" in line.lower():
                if "required" in line.lower():
                    current_term["importance"] = "required"
                elif "optional" in line.lower():
                    current_term["importance"] = "optional"

        if current_term and "term" in current_term:
            terms.append(current_term)

        logger.info(f"EntityExtractor: Fallback parser extracted {len(terms)} terms")
        return terms

    def __call__(self, state):
        """Make the class callable for langgraph."""
        logger.info("EntityExtractor node called")
        result = self.extract_entities_properties(state)
        logger.info("EntityExtractor node completed")
        return result

class QueryGenerator:
    """Node for generating SPARQL queries from entities and properties."""

    def __init__(self, llm):
        logger.info("Initializing QueryGenerator")
        self.llm = llm

    def generate_query(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Generate a SPARQL query from entities and properties

        Args:
            state: Current state with 'entities' and 'properties' keys

        Returns:
            Updated state with generated SPARQL query
        """
        question = state.get("question", "")
        entities = state.get("entities", [])
        properties = state.get("properties", [])
        feedback = state.get("feedback", "")

        logger.info(f"QueryGenerator: Generating SPARQL query for question: '{question}'")
        logger.info(f"QueryGenerator: Using {len(entities)} entities and {len(properties)} properties")
        
        if feedback:
            logger.info(f"QueryGenerator: Incorporating feedback: '{feedback}'")

        # Prepare entity and property information for the prompt
        entity_info = []
        for entity in entities:
            entity_str = f"Entity: {entity.get('label')} (ID: {entity.get('id')}), Description: {entity.get('description')}"
            logger.debug(f"QueryGenerator: Entity info: {entity_str}")
            entity_info.append(entity_str)

        property_info = []
        for prop in properties:
            prop_str = f"Property: {prop.get('label')} (ID: {prop.get('id')}), Description: {prop.get('description')}"
            logger.debug(f"QueryGenerator: Property info: {prop_str}")
            property_info.append(prop_str)

        entity_info_str = "\n".join(entity_info)
        property_info_str = "\n".join(property_info)

        prompt = f"""
        Generate a SPARQL query for the Wikidata endpoint that answers the following question:
        
        Question: {question}
        
        Using these entities and properties:
        {entity_info_str}
        {property_info_str}
        
        IMPORTANT RULES:
        1. The query should ONLY return URIs, NOT labels.
        2. Do NOT use rdfs:label, wikibase:label, or SERVICE wikibase:label.
        3. Do NOT include variables with "Label" suffix in the SELECT clause.
        4. Use the appropriate Wikidata prefixes (wd, wdt, p, ps, etc.)
        5. Only use the entities and properties provided above.
        
        {"Previous feedback to address: " + feedback if feedback else ""}
        
        Format your response as:
        ```sparql
        [YOUR SPARQL QUERY HERE]
        ```
        """

        logger.info("QueryGenerator: Calling Qwen model to generate SPARQL query")
        raw_response = self.llm.invoke(prompt)
        logger.debug(f"QueryGenerator: Raw model response: {raw_response[:500]}...")

        # Try to extract query from code block
        sparql_pattern = r"```(?:sparql)?\s*([\s\S]*?)```"
        match = re.search(sparql_pattern, raw_response)
        
        if match:
            logger.info("QueryGenerator: Extracted SPARQL query from code block")
            query = match.group(1).strip()
        else:
            # If no code block found, use the entire response
            logger.info("QueryGenerator: No code block found, using entire response as query")
            query = raw_response

        logger.info("QueryGenerator: SPARQL query generated successfully")
        logger.info(f"QueryGenerator: Generated query:\n{query}")

        return {**state, "generated_query": query, "generation_complete": True}

    def __call__(self, state):
        """Make the class callable for langgraph."""
        logger.info("QueryGenerator node called")
        result = self.generate_query(state)
        logger.info("QueryGenerator node completed")
        return result

class QueryChecker:
    """Node for checking and validating SPARQL queries."""

    def __init__(self, llm):
        logger.info("Initializing QueryChecker")
        self.llm = llm
        self.sparql_tool = WikidataSPARQLTool()

    def check_query(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """
        Check a SPARQL query and decide whether to use it or regenerate

        Args:
            state: Current state with 'generated_query' key

        Returns:
            Updated state with query validation results and decision
        """
        question = state.get("question", "")
        query = state.get("generated_query", "")

        logger.info(f"QueryChecker: Checking query for question: '{question}'")
        logger.info(f"QueryChecker: Query to check:\n{query}")

        # Execute the query
        logger.info("QueryChecker: Executing SPARQL query against Wikidata")
        result = self.sparql_tool.execute(query)

        success = result.get("success", False)
        result_count = result.get("count", 0)
        results = result.get("results", [])

        if success:
            logger.info(f"QueryChecker: Query executed successfully. Result count: {result_count}")
        else:
            error = result.get("error", "Unknown error")
            logger.error(f"QueryChecker: Query execution failed. Error: {error}")

        # Get labels for URIs in results for better human-readable feedback
        all_uris = []
        for res in results:
            for key, value in res.items():
                if isinstance(value, str) and value.startswith("http://www.wikidata.org/entity/"):
                    all_uris.append(value)

        if all_uris:
            logger.info(f"QueryChecker: Getting labels for {len(all_uris)} URIs")
            uri_labels = self.sparql_tool.get_labels_for_uris(all_uris)
            logger.debug(f"QueryChecker: Retrieved {len(uri_labels)} labels")
        else:
            logger.info("QueryChecker: No URIs to get labels for")
            uri_labels = {}

        # Add labels to results for context
        results_with_labels = []
        for res in results:
            res_with_labels = {}
            for key, value in res.items():
                res_with_labels[key] = value
                if value in uri_labels:
                    res_with_labels[f"{key}_label"] = uri_labels[value]
            results_with_labels.append(res_with_labels)

        # Log a sample of the results
        if results_with_labels:
            sample_size = min(3, len(results_with_labels))
            logger.info(f"QueryChecker: Sample of results (first {sample_size}):")
            for i, res in enumerate(results_with_labels[:sample_size]):
                logger.info(f"  Result {i+1}: {res}")
            if len(results_with_labels) > sample_size:
                logger.info(f"  ... and {len(results_with_labels) - sample_size} more results")

        # Check if the query is valid and results are satisfactory
        if not success:
            error = result.get("error", "Unknown error")
            feedback = (
                f"The query failed with error: {error}. Please fix the SPARQL syntax."
            )
            decision = "regenerate"
            logger.warning(f"QueryChecker: Query failed, will regenerate. Error: {error}")
        elif result_count == 0:
            feedback = "The query executed successfully but returned no results. Please adjust the query to return relevant results."
            decision = "regenerate"
            logger.warning("QueryChecker: Query returned no results, will regenerate")
        else:
            # Use Qwen to evaluate if the results actually answer the question
            logger.info("QueryChecker: Evaluating relevance of results using Qwen")
            
            evaluation_prompt = f"""
            Question: {question}
            
            SPARQL Query:
            {query}
            
            Query Results (first {min(5, len(results_with_labels))} of {result_count}):
            {results_with_labels[:5]}
            
            Do these results correctly answer the original question? Evaluate based on:
            1. Are the results relevant to the question?
            2. Do they contain the information needed to answer the question?
            3. Is the query constructed properly to capture the intent of the question?
            
            Respond with:
            - "satisfied" if the results adequately answer the question
            - "regenerate" if the query needs to be modified, with specific feedback on what's wrong
            
            Format your response as:
            DECISION: [satisfied or regenerate]
            FEEDBACK: [your feedback if regenerate, or "Results look good." if satisfied]
            """

            evaluation_text = self.llm.invoke(evaluation_prompt)
            logger.debug(f"QueryChecker: Qwen evaluation response: {evaluation_text}")

            # Parse the decision and feedback
            decision = "regenerate"  # Default
            feedback = ""

            if "DECISION:" in evaluation_text:
                decision_line = [
                    line for line in evaluation_text.split("\n") if "DECISION:" in line
                ][0]
                decision_text = decision_line.split("DECISION:")[1].strip().lower()
                if "satisfied" in decision_text:
                    decision = "satisfied"
                else:
                    decision = "regenerate"

            if "FEEDBACK:" in evaluation_text:
                feedback_start = evaluation_text.find("FEEDBACK:") + len("FEEDBACK:")
                feedback = evaluation_text[feedback_start:].strip()

            logger.info(f"QueryChecker: Evaluation decision: {decision}")
            logger.info(f"QueryChecker: Evaluation feedback: {feedback}")

        return {
            **state,
            "query_results": results_with_labels,
            "result_count": result_count,
            "query_success": success,
            "feedback": feedback,
            "decision": decision,
        }

    def decide_next_step(self, state: Dict[str, Any]) -> Literal["continue", "regenerate"]:
        """Determine the next step based on the query checker's decision."""
        decision = state.get("decision", "regenerate")
        
        if decision == "satisfied":
            logger.info("QueryChecker: Decision - continue (satisfied with results)")
            return "continue"
        else:
            logger.info("QueryChecker: Decision - regenerate (not satisfied with results)")
            return "regenerate"

    def __call__(self, state):
        """Make the class callable for langgraph."""
        logger.info("QueryChecker node called")
        result = self.check_query(state)
        logger.info("QueryChecker node completed")
        return result

## LangGraph State Definition

In [6]:
class WikidataQueryState(TypedDict):
    question: str
    entities: list
    properties: list
    generated_query: str
    query_results: list
    result_count: int
    query_success: bool
    feedback: str
    decision: Literal["satisfied", "regenerate"]
    final_query: str

## Wikidata Query Agent Implementation

In [7]:
def create_wikidata_graph(llm):
    """
    Create the Wikidata query graph
    
    Args:
        llm: Language model for node operations
        
    Returns:
        Configured graph for Wikidata querying
    """
    logger.info("Creating Wikidata query graph")
    
    # Initialize nodes
    logger.info("Initializing graph nodes")
    entity_extractor = EntityExtractor(llm)
    query_generator = QueryGenerator(llm)
    query_checker = QueryChecker(llm)
    
    # Define state graph
    graph = StateGraph(WikidataQueryState)
    
    # Define nodes
    graph.add_node("extract_entities", entity_extractor)
    graph.add_node("generate_query", query_generator)
    graph.add_node("check_query", query_checker)
    
    # Define edges
    graph.add_edge("extract_entities", "generate_query")
    graph.add_edge("generate_query", "check_query")
    
    # Define conditional edges for the checker
    graph.add_conditional_edges(
        "check_query",
        query_checker.decide_next_step,
        {
            "continue": END,
            "regenerate": "generate_query"
        }
    )
    
    # Define the entry point
    graph.set_entry_point("extract_entities")
    
    logger.info("Graph creation complete")
    
    # Compile the graph
    return graph.compile()

class WikidataQueryAgent:
    """Agent for answering questions by querying Wikidata using SPARQL."""
    
    def __init__(self, llm):
        logger.info("Initializing WikidataQueryAgent")
        self.graph = create_wikidata_graph(llm)
    
    def query(self, question: str) -> Dict[str, Any]:
        """
        Process a question and generate a SPARQL query
        
        Args:
            question: The natural language question
            
        Returns:
            Dictionary with the query, results, and other information
        """
        logger.info(f"Processing question: {question}")
        
        # Initialize state
        state = {
            "question": question,
            "entities": [],
            "properties": [],
            "generated_query": "",
            "query_results": [],
            "result_count": 0,
            "query_success": False,
            "feedback": "",
            "decision": "regenerate"
        }
        
        # Run the graph
        logger.info("Invoking LangGraph execution")
        result = self.graph.invoke(state)
        
        logger.info(f"Graph execution complete. Query success: {result.get('query_success', False)}")
        
        # Prepare the final result
        final_result = {
            "question": question,
            "sparql_query": result.get("generated_query", ""),
            "results": result.get("query_results", []),
            "result_count": result.get("result_count", 0),
            "success": result.get("query_success", False)
        }
        
        return final_result

## Evaluation Function

In [8]:
def compare_two_dataframes(df1: pd.DataFrame, df2: pd.DataFrame) -> Dict[str, float]:
    """
    Compare two dataframes and calculate various metrics.
    df1: DataFrame for ground truth
    df2: DataFrame for predicted
    """
    logger.info(f"Comparing dataframes: ground truth shape {df1.shape}, predicted shape {df2.shape}")
    
    if len(df1.columns) != len(df2.columns):
        logger.warning(f"Column count mismatch: ground truth {len(df1.columns)}, predicted {len(df2.columns)}")
        return {
            'jaccard': 0,
            'recall': 0,
            'precision': 0,
            'f1': 0,
            'tp': 0,
            'fp': 0,
            'fn': 0,
            'tn': 0
        }

    set1, set2 = set(), set()
    for _, row in df1.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set1.add(row)

    for _, row in df2.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set2.add(row)
    
    logger.debug(f"Converted to sets: ground truth {len(set1)} unique rows, predicted {len(set2)} unique rows")
    
    jaccard = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0
    # recall = correct retrieved / all ground truth
    recall = len(set1 & set2) / len(set1) if len(set1) > 0 else 0
    # precision = correct retrieved / retrieved answers
    precision = len(set1 & set2) / len(set2) if len(set2) > 0 else 0
    # f1 score = 2 x prec x recall / (prec + recall)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # TP, TN, FP, FN computation (might be useful for computing micro metrics)
    tp = len(set1 & set2)
    fp = len(set2) - tp
    fn = len(set1) - tp
    total_pairs = len(set1) + len(set2) - tp
    tn = total_pairs - (tp + fp + fn)

    metrics = {
        'jaccard': jaccard,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }
    
    logger.info(f"Metrics: jaccard={jaccard:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")
    logger.debug(f"Detailed metrics: tp={tp}, fp={fp}, fn={fn}, tn={tn}")
    
    return metrics

def execute_sparql_to_df(query: str) -> pd.DataFrame:
    """Execute a SPARQL query and convert results to DataFrame"""
    logger.info(f"Executing SPARQL query to DataFrame")
    logger.debug(f"Query: {query}")
    
    sparql_tool = WikidataSPARQLTool()
    result = sparql_tool.execute(query)
    
    if result.get('success', False) and 'results' in result:
        # Convert results to DataFrame
        df = pd.DataFrame(result['results'])
        logger.info(f"Query execution successful: {df.shape[0]} rows, {df.shape[1]} columns")
        return df
    else:
        # Return empty DataFrame on error
        error = result.get('error', 'Unknown error')
        logger.error(f"Query execution failed: {error}")
        return pd.DataFrame()

def evaluate_wikidata_agent(agent: WikidataQueryAgent, test_data_path: str, output_log_path: str = "evaluation_results.json"):
    """Evaluate the Wikidata Agent against a test dataset"""
    
    # Load test data
    logger.info(f"Loading test data from {test_data_path}")
    with open(test_data_path, 'r') as f:
        test_data = json.load(f)
    
    logger.info(f"Loaded {len(test_data)} test items")
    
    # For testing, limit to first few examples
    # Adjust this number based on available resources and time constraints
    test_limit = 10
    test_data = test_data[:test_limit]
    logger.info(f"Using first {len(test_data)} test items for evaluation")
    
    # Prepare results storage
    results = []
    metrics_sum = {
        'jaccard': 0,
        'recall': 0,
        'precision': 0,
        'f1': 0,
        'tp': 0,
        'fp': 0,
        'fn': 0,
        'tn': 0
    }
    
    # Process each test question
    for i, test_item in tqdm(enumerate(test_data), total=len(test_data), desc="Evaluating"):
        question = test_item['question']
        ground_truth_query = test_item['sparql']
        
        logger.info(f"Processing question {i+1}/{len(test_data)}: {question}")
        print(f"\nProcessing question {i+1}/{len(test_data)}: {question}")
        
        try:
            # Generate query using the agent
            logger.info("Generating query using agent")
            result = agent.query(question)
            generated_query = result['sparql_query']
            logger.info(f"Agent generated query: {generated_query}")
            print(f"Generated query:\n{generated_query}")
            
            # Execute both queries to get dataframes
            logger.info("Executing ground truth query...")
            ground_truth_df = execute_sparql_to_df(ground_truth_query)
            
            logger.info("Executing generated query...")
            generated_df = execute_sparql_to_df(generated_query)
            
            # Compare results
            logger.info("Comparing query results")
            metrics = compare_two_dataframes(ground_truth_df, generated_df)
            
            # Update metrics sum
            for key in metrics_sum:
                metrics_sum[key] += metrics[key]
            
            # Save individual result
            result_item = {
                'question': question,
                'ground_truth_query': ground_truth_query,
                'generated_query': generated_query,
                'metrics': metrics,
                'success': True
            }
            
            print(f"Metrics: Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}")
            
        except Exception as e:
            logger.error(f"Error processing question: {str(e)}", exc_info=True)
            print(f"Error: {str(e)}")
            result_item = {
                'question': question,
                'ground_truth_query': ground_truth_query,
                'error': str(e),
                'success': False
            }
            
            # Add zeros to metrics for failed queries
            for key in metrics_sum:
                metrics_sum[key] += 0
        
        results.append(result_item)
        
        # Print progress and current average metrics
        if (i + 1) % 5 == 0 or i == len(test_data) - 1:
            avg_metrics = {k: v / (i + 1) for k, v in metrics_sum.items()}
            logger.info(f"Current average metrics after {i + 1}/{len(test_data)} questions:")
            logger.info(f"Precision: {avg_metrics['precision']:.4f}, Recall: {avg_metrics['recall']:.4f}, F1: {avg_metrics['f1']:.4f}")
            
            print(f"\nCurrent average metrics after {i + 1}/{len(test_data)} questions:")
            print(f"Precision: {avg_metrics['precision']:.4f}")
            print(f"Recall: {avg_metrics['recall']:.4f}")
            print(f"F1: {avg_metrics['f1']:.4f}")
            print(f"Jaccard: {avg_metrics['jaccard']:.4f}")
    
    # Calculate average metrics
    avg_metrics = {k: v / len(test_data) for k, v in metrics_sum.items()}
    logger.info(f"Final average metrics: {avg_metrics}")
    
    # Save results
    final_results = {
        'results': results,
        'average_metrics': avg_metrics
    }
    
    logger.info(f"Saving evaluation results to {output_log_path}")
    with open(output_log_path, 'w') as f:
        json.dump(final_results, f, indent=2)
    
    return final_results

## Main Execution

In [9]:
import os

# Create dataset directory if it doesn't exist
!mkdir -p dataset/qald_9_plus

# Download test dataset
!wget -O dataset/qald_9_plus/qald_9_plus_test_wikidata.json https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/version_2.2.eval/dataset/qald_9_plus/qald_9_plus_test_wikidata.json

# Verify download
!ls -la dataset/qald_9_plus/

--2025-04-26 05:38:06--  https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/version_2.2.eval/dataset/qald_9_plus/qald_9_plus_test_wikidata.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30661 (30K) [text/plain]
Saving to: ‘dataset/qald_9_plus/qald_9_plus_test_wikidata.json’

dataset/qald_9_plus 100%[===================>]  29.94K  --.-KB/s    in 0s      

2025-04-26 05:38:07 (102 MB/s) - ‘dataset/qald_9_plus/qald_9_plus_test_wikidata.json’ saved [30661/30661]

total 40
drwxr-xr-x 2 root root  4096 Apr 26 05:38 .
drwxr-xr-x 3 root root  4096 Apr 26 05:38 ..
-rw-r--r-- 1 root root 30661 Apr 26 05:38 qald_9_plus_test_wikidata.json


In [10]:
# Initialize the Qwen model
llm = create_qwen_llm(max_seq_length=4096, load_in_4bit=True)

Initializing Qwen 2.5 7B 4-bit model...
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.1.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/106k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Device set to use cuda:0


Qwen model initialized successfully.


/tmp/ipykernel_31/832068843.py:37: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)


In [11]:
# Initialize the agent
print("Initializing Wikidata Query Agent with Qwen 7B model...")
agent = WikidataQueryAgent(llm)

Initializing Wikidata Query Agent with Qwen 7B model...


In [13]:
# Run evaluation
test_data_path = "dataset/qald_9_plus/qald_9_plus_test_wikidata.json"
print(f"Evaluating agent on {test_data_path}...")
results = evaluate_wikidata_agent(agent, test_data_path, "qwen_evaluation_results.json")

Evaluating agent on dataset/qald_9_plus/qald_9_plus_test_wikidata.json...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]


Processing question 1/10: What is the time zone of Salt Lake City?


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Evaluating:  10%|█         | 1/10 [00:31<04:40, 31.12s/it]

Error: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

Processing question 2/10: Who killed Caesar?


Evaluating:  20%|██        | 2/10 [01:33<06:34, 49.35s/it]

Error: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

Processing question 3/10: What is the highest mountain in Germany?


Evaluating:  20%|██        | 2/10 [02:07<08:28, 63.55s/it]


KeyboardInterrupt: 

In [ ]:
# Print summary
print("\nEvaluation complete!")
print(f"Results saved to qwen_evaluation_results.json")
print("\nAverage Metrics:")
for key, value in results['average_metrics'].items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Analyze results and failures
failed_queries = [r for r in results['results'] if not r.get('success', False)]
successful_queries = [r for r in results['results'] if r.get('success', True)]

print(f"Total queries: {len(results['results'])}")
print(f"Successful queries: {len(successful_queries)}")
print(f"Failed queries: {len(failed_queries)}")

if failed_queries:
    print("\nFailed questions:")
    for i, result in enumerate(failed_queries, 1):
        print(f"{i}. {result['question']}")
        print(f"   Error: {result.get('error', 'Unknown error')}")
        print("")

In [ ]:
# Plot the metrics for successful queries
import matplotlib.pyplot as plt
import seaborn as sns

# Get metrics for successful queries
metrics_list = [result['metrics'] for result in successful_queries if 'metrics' in result]

if metrics_list:
    # Prepare data for plotting
    precision_values = [m['precision'] for m in metrics_list]
    recall_values = [m['recall'] for m in metrics_list]
    f1_values = [m['f1'] for m in metrics_list]
    jaccard_values = [m['jaccard'] for m in metrics_list]
    
    # Create a figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot histograms for each metric
    sns.histplot(precision_values, bins=10, kde=True, ax=axes[0, 0])
    axes[0, 0].set_title('Precision Distribution')
    axes[0, 0].set_xlabel('Precision')
    
    sns.histplot(recall_values, bins=10, kde=True, ax=axes[0, 1])
    axes[0, 1].set_title('Recall Distribution')
    axes[0, 1].set_xlabel('Recall')
    
    sns.histplot(f1_values, bins=10, kde=True, ax=axes[1, 0])
    axes[1, 0].set_title('F1 Score Distribution')
    axes[1, 0].set_xlabel('F1 Score')
    
    sns.histplot(jaccard_values, bins=10, kde=True, ax=axes[1, 1])
    axes[1, 1].set_title('Jaccard Similarity Distribution')
    axes[1, 1].set_xlabel('Jaccard Similarity')
    
    plt.tight_layout()
    plt.show()
    
    # Create a scatter plot of precision vs recall
    plt.figure(figsize=(10, 6))
    plt.scatter(recall_values, precision_values, alpha=0.7)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision vs Recall for Generated Queries')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.show()
else:
    print("No metrics available for plotting.")